# 📊 Phase 5 — Model Evaluation

**Project:** Facial Skin Condition Detection  
**Level:** MCA Academic Project  

---

## Overview

This notebook provides a comprehensive evaluation of all 7 trained ML models:

1. **Model Comparison** — Ranked table + bar chart
2. **Confusion Matrices** — For all 7 models
3. **Classification Report** — Per-class precision, recall, F1 for best model
4. **Feature Importance** — If supported by best model
5. **ROC Curves** — One-vs-Rest for best model
6. **Error Analysis** — Where does the model fail?

In [ ]:
import sys, pickle, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, RocCurveDisplay,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import label_binarize

from evaluate_models import run_evaluation, compare_models, plot_confusion_matrices, generate_classification_report

MODELS_DIR  = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

print('Imports OK')

## 1. Load Model Results

In [ ]:
pkl_path = MODELS_DIR / 'model_results.pkl'
if not pkl_path.exists():
    print('⚠ model_results.pkl not found. Run notebook 04 first.')
else:
    with open(pkl_path, 'rb') as f:
        payload = pickle.load(f)

    model_results = payload['model_results']
    best_name     = payload['best_model_name']
    X_test        = payload['X_test']
    y_test        = payload['y_test']
    le            = payload['label_encoder']
    best_model    = payload['best_model']
    class_names   = le.classes_

    print(f'Best model  : {best_name}')
    print(f'Test samples: {len(y_test)}')
    print(f'Classes     : {list(class_names)}')

## 2. Run Full Evaluation Pipeline

In [ ]:
if 'payload' in dir():
    run_evaluation()

## 3. Best Model — Detailed Classification Report

In [ ]:
if 'payload' in dir():
    y_pred_best = model_results[best_name]['y_pred']
    report_dict = classification_report(
        y_test, y_pred_best,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).T
    print(f'Classification Report — {best_name}\n')
    report_df.round(3)

## 4. ROC Curves (One-vs-Rest)

In [ ]:
if 'payload' in dir() and hasattr(best_model, 'predict_proba'):
    y_score = best_model.predict_proba(X_test)
    y_bin   = label_binarize(y_test, classes=range(len(class_names)))

    palette = ['#E74C3C', '#E67E22', '#9B59B6', '#F1C40F', '#2ECC71']

    fig, ax = plt.subplots(figsize=(9, 7))
    for i, (cls, col) in enumerate(zip(class_names, palette)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=col, linewidth=2,
                label=f'{cls} (AUC = {roc_auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'ROC Curves (One-vs-Rest) — {best_name}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'roc_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved → reports/roc_curves.png')
else:
    if 'payload' in dir():
        print(f'{best_name} does not support predict_proba — ROC not available.')

## 5. Feature Importance (if supported)

In [ ]:
if 'payload' in dir() and hasattr(best_model, 'feature_importances_'):
    feat_cols  = payload.get('feature_cols', [f'F{i}' for i in range(58)])
    importances = best_model.feature_importances_
    indices     = np.argsort(importances)[::-1][:20]
    top_feats   = [feat_cols[i] for i in indices]
    top_vals    = importances[indices]

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top_feats[::-1], top_vals[::-1], color='#3498DB', edgecolor='white')
    ax.set_xlabel('Importance Score', fontsize=12)
    ax.set_title(f'Top 20 Feature Importances — {best_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved → reports/feature_importance.png')
else:
    if 'payload' in dir():
        print(f'{best_name} does not support feature importances.')

## 6. Model Summary Table

In [ ]:
if 'model_results' in dir():
    rows = []
    for name, res in model_results.items():
        clf_rep = res['clf_report']
        rows.append({
            'Model':         name,
            'CV Acc':        round(res['cv_mean'], 4),
            'CV Std':        round(res['cv_std'],  4),
            'Test Acc':      round(res['test_accuracy'], 4),
            'Macro F1':      round(clf_rep.get('macro avg', {}).get('f1-score', 0), 4),
            'Weighted F1':   round(clf_rep.get('weighted avg', {}).get('f1-score', 0), 4),
        })

    summary_df = pd.DataFrame(rows).sort_values('CV Acc', ascending=False).reset_index(drop=True)
    summary_df.index += 1
    summary_df

---
## Final Report

All evaluation charts saved to `reports/`:
- `model_comparison.png`
- `confusion_matrices.png`
- `roc_curves.png`
- `feature_importance.png` (if applicable)

**Next:** Run the Flask web application:
```bash
cd app
python app.py
```
Then open: http://localhost:5000